# Train YOLO11n on ACDC — Improved Baseline

This notebook trains a **YOLO11n** model with optimized hyperparameters:
- **640px** image size (vs 512px in baseline)
- **100 epochs** with cosine LR scheduler
- **AdamW optimizer** with warmup
- **Stronger augmentation**: mosaic, mixup, copy-paste
- **No layer freezing** (full fine-tuning)
- **Per-class evaluation** after training

Expected improvement: **+5-10 mAP@0.5** over the YOLOv8n baseline (0.2664).

In [ ]:
# Cell 1: Setup — Mount Drive, clone repo, install dependencies
import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Clone or pull the repo
REPO_DIR = Path('/content/ETH_AI_OBJ_DETECTION')
if REPO_DIR.exists():
    print('Repo exists, pulling latest...')
    os.system('cd /content/ETH_AI_OBJ_DETECTION && git pull')
else:
    print('Cloning repo...')
    os.system('git clone https://github.com/Danielit707/ETH_AI_OBJ_DETECTION.git /content/ETH_AI_OBJ_DETECTION')

!pip install -q ultralytics

from ultralytics import YOLO
import torch
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/acdc_data')
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/acdc_training_output')
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path('/content/acdc_data')
PROCESSED_DIR = Path('/content/acdc_processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: Extract ACDC data from Google Drive
import shutil
import zipfile

def safe_extract(zip_path, extract_to, allowed_prefixes=None):
    """Extract ZIP with path-traversal protection, only adverse-condition train/val."""
    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)
    conditions = ['fog', 'night', 'rain', 'snow']
    splits = ['train', 'val']
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.namelist():
            parts = member.replace('\\', '/').split('/')
            if len(parts) < 2:
                continue
            condition = parts[0].lower()
            split = parts[1].lower() if len(parts) > 2 else ''
            if condition in conditions and split in splits:
                dest = extract_to / member
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    with zf.open(member) as src, open(dest, 'wb') as dst:
                        shutil.copyfileobj(src, dst)

annotation_zip = DRIVE_DATA_ROOT / 'gt_detection_trainval.zip'
image_zip = DRIVE_DATA_ROOT / 'rgb_anon_trainvaltest.zip'

print('Extracting annotations...')
safe_extract(annotation_zip, DATA_DIR / 'annotations')
print('Extracting images...')
safe_extract(image_zip, DATA_DIR / 'images')
print('Done!')

In [ ]:
# Cell 3: Convert COCO to YOLO format
import subprocess

result = subprocess.run(
    [
        sys.executable, 'dataset/run_conversion.py',
        '--acdc-root', str(DATA_DIR),
        '--output-dir', str(PROCESSED_DIR),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Conversion failed')

# Verify the output
print('Converted dataset structure:')
for split in ['train', 'val']:
    images = list((PROCESSED_DIR / 'images' / split).glob('*'))
    labels = list((PROCESSED_DIR / 'labels' / split).glob('*'))
    print(f'  {split}: {len(images)} images, {len(labels)} labels')

In [ ]:
# Cell 4: Train YOLO11n with optimized hyperparameters
import yaml

# Create optimized data.yaml
data_yaml = {
    'path': str(PROCESSED_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'person',
        1: 'rider',
        2: 'car',
        3: 'truck',
        4: 'bus',
        5: 'train',
        6: 'motorcycle',
        7: 'bicycle',
    },
}

yaml_path = PROCESSED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print(f'data.yaml written to {yaml_path}')

# Load YOLO11n (pretrained on COCO)
model = YOLO('yolov11n.pt')
print(f'Loaded YOLO11n pretrained model')

# Train with optimized config
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    device=0,
    project=str(DRIVE_CHECKPOINT_DIR),
    name='yolov11n_acdc',
    exist_ok=True,
    plots=True,
    val=True,
    # Sync to Drive at each epoch
    callbacks={
        'on_fit_epoch_end': lambda trainer: (
            shutil.copy2(
                Path(trainer.save_dir) / 'last.pt',
                DRIVE_CHECKPOINT_DIR / 'last.pt'
            ),
            shutil.copy2(
                Path(trainer.save_dir) / 'best.pt',
                DRIVE_CHECKPOINT_DIR / 'best.pt'
            ),
        )
    },
)

print('Training complete!')
print(f'Best model: {DRIVE_CHECKPOINT_DIR / "best.pt"}')

In [ ]:
# Cell 5: Evaluate and compare with baseline
import json

eval_model = YOLO(str(DRIVE_CHECKPOINT_DIR / 'best.pt'))
eval_results = eval_model.val(
    data=str(yaml_path),
    imgsz=640,
    conf=0.001,
    plots=True,
    project=str(DRIVE_CHECKPOINT_DIR / 'evaluation'),
    exist_ok=True,
    device=0,
)

names = eval_results.names if isinstance(eval_results.names, dict) else {
    i: n for i, n in enumerate(eval_results.names)
}

summary = {
    'model': 'YOLO11n',
    'model_path': str(DRIVE_CHECKPOINT_DIR / 'best.pt'),
    'data': str(yaml_path),
    'imgsz': 640,
    'metrics': {
        'precision': round(float(eval_results.box.mp), 4),
        'recall': round(float(eval_results.box.mr), 4),
        'mAP@0.5': round(float(eval_results.box.map50), 4),
        'mAP@0.5:0.95': round(float(eval_results.box.map), 4),
    },
    'class_names': {str(k): v for k, v in names.items()},
}

metrics_file = DRIVE_CHECKPOINT_DIR / 'evaluation' / 'metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 60)
print('YOLO11n EVALUATION RESULTS')
print('=' * 60)
print(f"  Precision:     {summary['metrics']['precision']}")
print(f"  Recall:        {summary['metrics']['recall']}")
print(f"  mAP@0.5:       {summary['metrics']['mAP@0.5']}")
print(f"  mAP@0.5:0.95:  {summary['metrics']['mAP@0.5:0.95']}")
print()
print('Baseline (YOLOv8n):  mAP@0.5 = 0.2664, mAP@0.5:0.95 = 0.1538')
print(f"Improvement:         mAP@0.5 {'+' if summary['metrics']['mAP@0.5'] >= 0.2664 else ''}{summary['metrics']['mAP@0.5'] - 0.2664:.4f}, mAP@0.5:0.95 {'+' if summary['metrics']['mAP@0.5:0.95'] >= 0.1538 else ''}{summary['metrics']['mAP@0.5:0.95'] - 0.1538:.4f}")
print()
print(f'Metrics saved to: {metrics_file}')
print(f'Copy best.pt to your local models/ folder and update MODEL_PATH!')